# 创建一个 Llava 模型

本 notebook 将基于 clip-vit-large-patch-14-336 和 Qwen1.5-4b-Chat 模型，搭建 Llava-1.5 架构。

In [ ]:
# pip install transformers==4.57.6 huggingface_hub modelscope

## 下载模型

和 [使用Llava-1.5-7b.ipynb](使用Llava-1.5-7b.ipynb) 中一样，我们先下载好模型。如果镜像站下载太慢，建议使用魔塔社区下载。

```bash

pip install -U huggingface_hub


export HF_ENDPOINT=https://hf-mirror.com

huggingface-cli download --resume-download openai/clip-vit-large-patch14-336 --local-dir openai/clip-vit-large-patch14-336 --local-dir-use-symlinks False


huggingface-cli download --resume-download Qwen/Qwen1.5-4B-Chat --local-dir Qwen1.5-4B-Chat --local-dir-use-symlinks False
```

### 使用魔塔社区下载

In [ ]:
use_modelscope=True

In [ ]:
if use_modelscope:
    # 安装或更新 modelscope 库（如果环境中没有的话）
    # pip install -U modelscope
    
    # 从魔搭下载模型
    from modelscope import snapshot_download
    
    model_dir = snapshot_download(
        'AI-ModelScope/clip-vit-large-patch14-336',       # 魔搭上的模型 ID，与你给的链接一致
        cache_dir='/root/autodl-tmp',      # 强烈建议指向数据盘，避免系统盘被占满
        revision='master'                  # 主分支，通常就是最新的稳定版
    )
    
    print(f"vision 模型已下载到: {model_dir}")
    
    model_dir = snapshot_download(
        'Qwen/Qwen1.5-4B-Chat',       # 魔搭上的模型 ID，与你给的链接一致
        cache_dir='/root/autodl-tmp',      # 强烈建议指向数据盘，避免系统盘被占满
        revision='master'                  # 主分支，通常就是最新的稳定版
    )
        
    print(f"LLM 模型已下载到: {model_dir}")

## 修改 Qwen tokenizer

原始的 LLM 中是不包含 Llava 中引入的 `<image>` 词元的，需要我们手动添加到下载的文件中的`tokenizer_config.json`、中：

1. 在`added_tokens_decoder`里面，加上：
   ```json
    "151646": {
        "content": "<image>",
        "lstrip": false,
        "normalized": false,
        "rstrip": false,
        "single_word": false,
        "special": true
        }
    ```
2. 在`additional_special_tokens` 里面加上 `"<image>"`
   
添加之后简单做一下测试：